给定序列："ababc"
词汇表：{a, b, c}，大小 V = 3

1. 统计相邻转移次数

a→b 出现 2 次

b→a 出现 1 次

b→c 出现 1 次

从 b 出发的总次数 = 1 + 1 = 2

2. 加1平滑公式
P( x' | b ) = ( count(b→x') + 1 ) / ( count(b) + V )

3. 代入计算

P( a | b ) = ( 1 + 1 ) / ( 2 + 3 ) = 2 / 5

P( c | b ) = ( 1 + 1 ) / ( 2 + 3 ) = 2 / 5

最终答案：P(a|b) = 2/5 ，P(c|b) = 2/5

In [1]:
import re
from collections import Counter

def preprocess_text(text, n):
    # 1. 清洗
    text = re.sub(r'[^a-z\s]', '', text.lower())
    tokens = text.split()
    
    # 2. 构建词汇表（按频率降序）
    freq = Counter(tokens)
    sorted_words = sorted(freq.items(), key=lambda x: (-x[1], x[0]))
    word2id = {word: idx for idx, (word, _) in enumerate(sorted_words)}
    
    # 3. 滑动窗口
    features, labels = [], []
    for i in range(len(tokens) - n):
        window = tokens[i : i+n]
        if i + n < len(tokens):
            features.append([word2id[w] for w in window])
            labels.append(word2id[tokens[i+n]])
    # 注意：题目示例中保留了无标签窗口，但说明“若无后续词则忽略”，故此处只保留有标签的。
    return word2id, (features, labels)

模型定义
h_t = W_hh * h_{t-1} + W_hx * x_t
o_t = W_oh * h_t
损失函数 L = 1/2 * Σ_{t=1 到 T} ( o_t - y_t )²

梯度推导过程
令误差项 δ_t = ( o_t - y_t ) * W_oh^T
则损失对 W_hh 的梯度为：

∂L/∂W_hh = Σ_{t=1 到 T} Σ_{s=1 到 t} δ_t * ( Π_{k=s+1 到 t} W_hh ) * h_{s-1}^T

梯度消失与爆炸的条件

若 W_hh 的谱半径（最大特征值的绝对值）小于 1，则连乘项 Π W_hh 随 t-s 增大指数衰减 → 梯度消失

若 W_hh 的谱半径大于 1，则连乘项指数增长 → 梯度爆炸

In [2]:
import numpy as np

def rnn_forward(x, prev_h, W_hh, W_hx, b):
    pre = prev_h @ W_hh + x @ W_hx + b
    h = np.tanh(pre)
    return h, pre  # 保存 pre 用于反向

def rnn_backward(dh_next, x, prev_h, h, pre, W_hh, W_hx, b):
    # dh_next 形状 (batch, hidden)
    dpre = dh_next * (1 - np.tanh(pre)**2)  # 元素乘
    # 梯度
    dW_hh = prev_h.T @ dpre            # (hidden, hidden)
    dW_hx = x.T @ dpre                 # (input, hidden)
    db = np.sum(dpre, axis=0)          # (hidden,)
    dh_prev = dpre @ W_hh.T            # (batch, hidden)
    dx = dpre @ W_hx.T                 # (batch, input)
    return dx, dh_prev, dW_hh, dW_hx, db

第 1 层（双向）
前向和后向各包含：输入→隐藏(D×H) + 隐藏→隐藏(H×H) + 偏置(H)
单侧参数 = DH + H² + H
第 1 层总参数 P₁ = 2 * ( DH + H² + H )

第 l 层（l ≥ 2，双向）
输入来自上一层双向拼接，维度为 2H
单侧参数 = (2H)*H + H² + H = 3H² + H
第 l 层总参数 P_l = 2 * ( 3H² + H ) = 6H² + 2H

模型总参数
P_total = P₁ + (L - 1) * P_l
= 2DH + 2H² + 2H + (L - 1) * ( 6H² + 2H )

In [3]:
import torch
import torch.nn as nn

class BidirectionalRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.rnn = nn.RNN(input_dim, hidden_dim, num_layers,
                          batch_first=False, bidirectional=True)
        self.hidden_dim = hidden_dim

    def forward(self, X):
        # X: (seq_len, batch, input_dim)
        outputs, h_n = self.rnn(X)  # outputs: (seq_len, batch, 2*hidden_dim)
        # h_n: (num_layers*2, batch, hidden_dim)
        # 取最后一层的前向和后向最后一个时间步
        forward_last = h_n[-2, :, :]   # (batch, hidden)
        backward_last = h_n[-1, :, :]  # (batch, hidden)
        final_state = torch.cat([forward_last, backward_last], dim=-1)  # (batch, 2*hidden)
        # 为了与 inputs 保持一致，将 final_state 扩展为 (seq_len, batch, 2*hidden) 的最后一个时间步？题目要求“最终时间步的拼接隐藏状态”，可以取 outputs[-1]？
        # 实际上 outputs[-1] 是前向最后 + 后向最后（但后向最后是反向RNN的初始状态），通常序列表示取前向最后 + 后向第一个。
        # 这里按常见做法：final_state = torch.cat([outputs[-1, :, :hidden_dim], outputs[0, :, hidden_dim:]], dim=-1)
        # 但我们直接使用 h_n 提取更准确，如上。
        # 但题目要求返回每个时间步的拼接状态，即 outputs，以及最终表示 final_state。
        return outputs, final_state

符号定义
中心词 w_c，词向量 v_c
上下文词 w_o，词向量 u_o
负样本集合 { n₁, n₂, ..., n_K }，词向量为 u_{n_k}

负采样损失函数（最小化形式）
J = - log σ( v_c^T · u_o ) - Σ_{k=1 到 K} log σ( - v_c^T · u_{n_k} )

其中 σ(x) = 1 / ( 1 + e^(-x) )

负样本采样方法
从噪声分布 P_n(w) 中独立采样 K 个负样本。
常用的噪声分布是 unigram 分布的 3/4 次方：
P_n(w) ∝ freq(w)^(3/4)
（即词的频率越高，被采为负样本的概率略降，低频词被采到的概率相对提升）

In [4]:
import numpy as np

def cbow_loss(context_indices, target_indices, W, W_out):
    batch_size, context_size = context_indices.shape
    V, d = W.shape
    
    # 1. 获取上下文嵌入并平均
    # context_indices: (batch, context_size) -> 嵌入 (batch, context_size, d)
    emb = W[context_indices]  # (batch, context_size, d)
    h = np.mean(emb, axis=1)  # (batch, d)
    
    # 2. 计算输出分数
    scores = h @ W_out  # (batch, V)
    
    # 3. softmax
    exp_scores = np.exp(scores - np.max(scores, axis=1, keepdims=True))
    probs = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
    
    # 4. 交叉熵损失
    loss = -np.log(probs[np.arange(batch_size), target_indices] + 1e-8)
    return np.mean(loss)

输入矩阵维度
Q : 2 × 4 （2 个查询）
K : 3 × 4 （3 个键）
V : 3 × 5 （3 个值）
d_k = 4 ，所以 √d_k = 2

计算步骤

Step 1：计算未缩放得分矩阵
S' = Q · K^T ，形状为 2 × 3

Step 2：缩放
S = S' / √d_k = ( Q · K^T ) / 2 ，形状仍为 2 × 3

Step 3：按行 Softmax（对每一行单独做）
对 S 的第 i 行（i=1,2），计算注意力权重：
A_ij = exp( S_ij ) / ( exp( S_i1 ) + exp( S_i2 ) + exp( S_i3 ) )
得到的 A 矩阵形状为 2 × 3

Step 4：加权求和得到输出
O = A · V ，形状为 2 × 5
即第 i 个查询的输出 = Σ_{j=1 到 3} A_ij * V_j （V_j 是 V 的第 j 行向量）

最终结果表达式
O = softmax( ( Q·K^T ) / 2 ) · V ，最终输出形状为 2 × 5

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, X):
        # X: (seq_len, batch, d_model)
        seq_len, batch, _ = X.shape
        
        # 线性投影并拆分为多头
        Q = self.W_q(X).view(seq_len, batch, self.num_heads, self.d_k).transpose(0, 2)  # (num_heads, seq_len, batch, d_k)
        K = self.W_k(X).view(seq_len, batch, self.num_heads, self.d_k).transpose(0, 2)
        V = self.W_v(X).view(seq_len, batch, self.num_heads, self.d_k).transpose(0, 2)
        
        # 计算缩放点积注意力（每个头独立）
        # Q: (num_heads, seq_len, batch, d_k), K: (num_heads, seq_len, batch, d_k)
        # 对每个头，计算 seq_len x seq_len 的注意力
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)  # (num_heads, seq_len, batch, seq_len)
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, V)  # (num_heads, seq_len, batch, d_k)
        
        # 拼接所有头
        out = out.transpose(0, 1).contiguous().view(seq_len, batch, self.d_model)  # (seq_len, batch, d_model)
        out = self.W_o(out)
        return out